## **회귀(regression) - 이어서**
### **06. 규제 선형 모델 - 릿지, 라쏘, 엘라스틱넷**
- 데이터는 책과 다르게 캘리포니아 데이터 이용

In [6]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# 데이터 로드
housing = fetch_california_housing()
housingDF = pd.DataFrame(housing.data, columns=housing.feature_names)
housingDF['PRICE'] = housing.target

# X, y 분리
X = housingDF.drop('PRICE', axis=1)
y = housingDF['PRICE']

# train/test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 스케일링 (규제 회귀 필수)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#### 1. 릿지 회귀

In [7]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

y_pred = ridge.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=== Ridge ===")
print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")

coef = pd.Series(ridge.coef_, index=X.columns).sort_values(ascending=False)
print("\n계수:")
print(coef)

=== Ridge ===
RMSE: 0.7456
R2: 0.5758

계수:
MedInc        0.854327
AveBedrms     0.339008
HouseAge      0.122624
Population   -0.002282
AveOccup     -0.040833
AveRooms     -0.294210
Longitude    -0.869071
Latitude     -0.896168
dtype: float64


#### 2. 라쏘 회귀

In [8]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.01)
lasso.fit(X_train_scaled, y_train)

y_pred = lasso.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=== Lasso ===")
print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")

coef = pd.Series(lasso.coef_, index=X.columns).sort_values(ascending=False)
print("\n계수:")
print(coef)

=== Lasso ===
RMSE: 0.7404
R2: 0.5816

계수:
MedInc        0.800957
AveBedrms     0.206207
HouseAge      0.127087
Population   -0.000000
AveOccup     -0.030602
AveRooms     -0.162759
Longitude    -0.755674
Latitude     -0.790113
dtype: float64


#### 3. 엘라스틱넷 회귀

In [10]:
from sklearn.linear_model import ElasticNet

elastic = ElasticNet(alpha=0.01, l1_ratio=0.5)
elastic.fit(X_train_scaled, y_train)

y_pred = elastic.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=== ElasticNet ===")
print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")

coef = pd.Series(elastic.coef_, index=X.columns).sort_values(ascending=False)
print("\n계수:")
print(coef)

=== ElasticNet ===
RMSE: 0.7416
R2: 0.5803

계수:
MedInc        0.823963
AveBedrms     0.256299
HouseAge      0.130005
Population   -0.000000
AveOccup     -0.035948
AveRooms     -0.216074
Longitude    -0.757036
Latitude     -0.788420
dtype: float64


In [11]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# 데이터 로드
housing = fetch_california_housing()
housingDF = pd.DataFrame(housing.data, columns=housing.feature_names)
housingDF['PRICE'] = housing.target

# X, y 분리
X = housingDF.drop('PRICE', axis=1)
y = housingDF['PRICE']

# 학습용 / 테스트용 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
from sklearn.linear_model import Ridge

# 시험해볼 alpha 후보
params = {
    'alpha': [0.01, 0.1, 1, 10, 100]
}

# Ridge 모델
ridge = Ridge()

# GridSearchCV: 5-fold 교차검증, 평가지표는 음수 MSE
grid_ridge = GridSearchCV(
    estimator=ridge,
    param_grid=params,
    scoring='neg_mean_squared_error',
    cv=5
)

# 최적 alpha 찾기
grid_ridge.fit(X_train_scaled, y_train)

# 최적 모델
best_ridge = grid_ridge.best_estimator_

# 테스트 데이터 예측
y_pred = best_ridge.predict(X_test_scaled)

# 성능 평가
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=== Ridge GridSearchCV ===")
print("Best alpha:", grid_ridge.best_params_['alpha'])
print(f"Best CV Score (neg MSE): {grid_ridge.best_score_:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test R2: {r2:.4f}")

# 회귀 계수
coef = pd.Series(best_ridge.coef_, index=X.columns).sort_values(ascending=False)
print("\n회귀 계수:")
print(coef)

=== Ridge GridSearchCV ===
Best alpha: 0.1
Best CV Score (neg MSE): -0.5193
Test RMSE: 0.7456
Test R2: 0.5758

회귀 계수:
MedInc        0.854377
AveBedrms     0.339234
HouseAge      0.122554
Population   -0.002305
AveOccup     -0.040829
AveRooms     -0.294390
Longitude    -0.869765
Latitude     -0.896853
dtype: float64


In [13]:
from sklearn.linear_model import Lasso

params = {
    'alpha': [0.001, 0.01, 0.1, 1, 10]
}

lasso = Lasso(max_iter=10000)

grid_lasso = GridSearchCV(
    estimator=lasso,
    param_grid=params,
    scoring='neg_mean_squared_error',
    cv=5
)

grid_lasso.fit(X_train_scaled, y_train)

best_lasso = grid_lasso.best_estimator_
y_pred = best_lasso.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=== Lasso GridSearchCV ===")
print("Best alpha:", grid_lasso.best_params_['alpha'])
print(f"Best CV Score (neg MSE): {grid_lasso.best_score_:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test R2: {r2:.4f}")

coef = pd.Series(best_lasso.coef_, index=X.columns).sort_values(ascending=False)
print("\n회귀 계수:")
print(coef)

=== Lasso GridSearchCV ===
Best alpha: 0.001
Best CV Score (neg MSE): -0.5192
Test RMSE: 0.7446
Test R2: 0.5769

회귀 계수:
MedInc        0.849140
AveBedrms     0.326050
HouseAge      0.123346
Population   -0.001062
AveOccup     -0.039890
AveRooms     -0.281273
Longitude    -0.858093
Latitude     -0.885822
dtype: float64


In [14]:
from sklearn.linear_model import ElasticNet

params = {
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

elastic = ElasticNet(max_iter=10000)

grid_elastic = GridSearchCV(
    estimator=elastic,
    param_grid=params,
    scoring='neg_mean_squared_error',
    cv=5
)

grid_elastic.fit(X_train_scaled, y_train)

best_elastic = grid_elastic.best_estimator_
y_pred = best_elastic.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=== ElasticNet GridSearchCV ===")
print("Best params:", grid_elastic.best_params_)
print(f"Best CV Score (neg MSE): {grid_elastic.best_score_:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test R2: {r2:.4f}")

coef = pd.Series(best_elastic.coef_, index=X.columns).sort_values(ascending=False)
print("\n회귀 계수:")
print(coef)

=== ElasticNet GridSearchCV ===
Best params: {'alpha': 0.001, 'l1_ratio': 0.9}
Best CV Score (neg MSE): -0.5192
Test RMSE: 0.7447
Test R2: 0.5768

회귀 계수:
MedInc        0.849584
AveBedrms     0.326979
HouseAge      0.123394
Population   -0.001145
AveOccup     -0.039990
AveRooms     -0.282283
Longitude    -0.857994
Latitude     -0.885673
dtype: float64


In [16]:
results = []

y_pred = best_ridge.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

results.append({
    'Model': 'Ridge',
    'Best Params': grid_ridge.best_params_,
    'RMSE': rmse,
    'R2': r2
})

y_pred = best_lasso.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

results.append({
    'Model': 'Lasso',
    'Best Params': grid_lasso.best_params_,
    'RMSE': rmse,
    'R2': r2
})

y_pred = best_elastic.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

results.append({
    'Model': 'ElasticNet',
    'Best Params': grid_elastic.best_params_,
    'RMSE': rmse,
    'R2': r2
})

result_df = pd.DataFrame(results)

# RMSE 기준으로 정렬 (작을수록 좋음)
result_df = result_df.sort_values(by='RMSE')

print(result_df)

        Model                        Best Params      RMSE        R2
1       Lasso                   {'alpha': 0.001}  0.744642  0.576856
2  ElasticNet  {'alpha': 0.001, 'l1_ratio': 0.9}  0.744697  0.576794
0       Ridge                     {'alpha': 0.1}  0.745579  0.575791


###
### **07. 로지스틱 회귀**

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression

cancer = load_breast_cancer()

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# StandardScaler( )로 평균이 0, 분산 1로 데이터 분포도 변환
scaler = StandardScaler()
data_scaled = scaler.fit_transform(cancer.data)

X_train , X_test, y_train , y_test = train_test_split(data_scaled, cancer.target, test_size=0.3, random_state=0)

In [3]:
from sklearn.metrics import accuracy_score, roc_auc_score

# 로지스틱 회귀를 이용하여 학습 및 예측 수행. 
# solver인자값을 생성자로 입력하지 않으면 solver='lbfgs'  
lr_clf = LogisticRegression() # solver='lbfgs'
lr_clf.fit(X_train, y_train)
lr_preds = lr_clf.predict(X_test)
lr_preds_proba = lr_clf.predict_proba(X_test)[:, 1]

# accuracy와 roc_auc 측정
print('accuracy: {0:.3f}, roc_auc:{1:.3f}'.format(accuracy_score(y_test, lr_preds),
                                                 roc_auc_score(y_test , lr_preds_proba)))

accuracy: 0.977, roc_auc:0.995


In [4]:
solvers = ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga']
# 여러개의 solver값 별로 LogisticRegression 학습 후 성능 평가
for solver in solvers:
    lr_clf = LogisticRegression(solver=solver, max_iter=600)
    lr_clf.fit(X_train, y_train)
    lr_preds = lr_clf.predict(X_test)
    lr_preds_proba = lr_clf.predict_proba(X_test)[:, 1]

    # accuracy와 roc_auc 측정
    print('solver:{0}, accuracy: {1:.3f}, roc_auc:{2:.3f}'.format(solver, 
                                                                  accuracy_score(y_test, lr_preds),
                                                                  roc_auc_score(y_test , lr_preds_proba)))                              

solver:lbfgs, accuracy: 0.977, roc_auc:0.995
solver:liblinear, accuracy: 0.982, roc_auc:0.995
solver:newton-cg, accuracy: 0.977, roc_auc:0.995
solver:sag, accuracy: 0.982, roc_auc:0.995
solver:saga, accuracy: 0.982, roc_auc:0.995


In [5]:
from sklearn.model_selection import GridSearchCV

params={'solver':['liblinear', 'lbfgs'],
        'penalty':['l2', 'l1'],
        'C':[0.01, 0.1, 1, 5, 10]}

lr_clf = LogisticRegression()

grid_clf = GridSearchCV(lr_clf, param_grid=params, scoring='accuracy', cv=3 )
grid_clf.fit(data_scaled, cancer.target)
print('최적 하이퍼 파라미터:{0}, 최적 평균 정확도:{1:.3f}'.format(grid_clf.best_params_, 
                                                  grid_clf.best_score_))

최적 하이퍼 파라미터:{'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}, 최적 평균 정확도:0.979


C:\Users\user\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
15 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\user\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\anaconda3\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\user\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py", line 1218, in fit
    solver = _chec